In [2]:
import sympy as sp
from IPython.display import Markdown, display

# ============================================================
# GENERIC LINEAR GAUSSIAN SYSTEM
# ============================================================

n = sp.symbols("n", positive=True, integer=True)

W = sp.MatrixSymbol("W", n, n)

mu_z = sp.MatrixSymbol("mu_z", n, 1)

z = sp.MatrixSymbol("z", n, 1)
y = sp.MatrixSymbol("y", n, 1)

Sigma_z = sp.MatrixSymbol("Sigma_z", n, n)
Sigma_y = sp.MatrixSymbol("Sigma_y", n, n)

Lambda_z = sp.MatrixSymbol("Lambda_z", n, n)
Lambda_y = sp.MatrixSymbol("Lambda_y", n, n)

b = sp.ZeroMatrix(n, 1)

# Gaussian "function"
Normal = sp.Function(r"\mathcal{N}")

# ============================================================
# Posterior
# ============================================================

Sigma_post = sp.Inverse(
    Lambda_z + W.T * Lambda_y * W
)

mu_post = Sigma_post * (
    W.T * Lambda_y * (y - b)
    + Lambda_z * mu_z
)

mu_marg = W * mu_z + b
Sigma_marg = Sigma_y + W * Sigma_z * W.T


# ============================================================
# PRINT MODEL GENERICALLY
# ============================================================
def printit(mapping):


    mapping[Lambda_y] = sp.Inverse(mapping[Sigma_y])
    mapping[Lambda_z] = sp.Inverse(mapping[Sigma_z])
    # Apply substitutions
    z_ex = z.subs(mapping)
    y_ex = y.subs(mapping)
    W_ex = W.subs(mapping)

    mu_z_ex = mu_z.subs(mapping)
    Sigma_z_ex = Sigma_z.subs(mapping)
    Sigma_y_ex = Sigma_y.subs(mapping)

    Sigma_post_ex = Sigma_post.subs(mapping)
    mu_post_ex = mu_post.subs(mapping)

    mu_marg_ex = mu_marg.subs(mapping)
    Sigma_marg_ex = Sigma_marg.subs(mapping)
    display(Markdown("# Model"))

    display(Markdown("### Prior"))

    display(
        sp.Eq(
            z_ex,
            Normal(mu_z_ex, Sigma_z_ex),
            evaluate=False
        )
    )

    display(Markdown("### Likelihood"))

    display(
        sp.Eq(
            y_ex,
            Normal(W_ex * z_ex + b, Sigma_y_ex),
            evaluate=False
        )
    )

    # ============================================================
    # PRINT POSTERIOR
    # ============================================================


    display(Markdown("# Posterior covariance eq. (3.37) Murphy1"))

    display(
        sp.Eq(
            sp.Symbol(rf"\Sigma_({y_ex}|{z_ex})"),
            Sigma_post_ex,
            evaluate=False
        )
    )

    display(Markdown("# Posterior mean eq. (3.37) Murphy1"))

    display(
        sp.Eq(
            sp.Symbol(rf"\mu_({y_ex}|{z_ex})"),
            mu_post_ex,
            evaluate=False
        )
    )

    display(Markdown("# Marginal eq. (3.38) Murphy1"))

    display(
        sp.Eq(
            y_ex,
            Normal(mu_marg_ex, Sigma_marg_ex),
            evaluate=False
        )
    )

# Part 1: Linear Gaussian systems

## Question 1.1: Determine the conditional distribution of x1 given x2

In [3]:
A = sp.MatrixSymbol("A", n, n)
Sigma = sp.MatrixSymbol("Sigma", n, n)

x1 = sp.MatrixSymbol("x_1", n, 1)
x2 = sp.MatrixSymbol("x_2", n, 1)
x3 = sp.MatrixSymbol("x_3", n, 1)

mapping = {
    y: x2,
    W: A,
    z: x1,
    Sigma_y: Sigma,
    mu_z: sp.ZeroMatrix(n, 1),
    Sigma_z: Sigma
}
printit(mapping)

# Model

### Prior

Eq(x_1, \mathcal{N}(0, Sigma))

### Likelihood

Eq(x_2, \mathcal{N}(A*x_1, Sigma))

...

Eq(\Sigma_(x_2|x_1), (Sigma**(-1) + A.T*Sigma**(-1)*A)**(-1))

# Posterior mean eq. (3.37) Murphy1

Eq(\mu_(x_2|x_1), (Sigma**(-1) + A.T*Sigma**(-1)*A)**(-1)*(Sigma**(-1)*0 + A.T*Sigma**(-1)*x_2))

# Marginal eq. (3.38) Murphy1

Eq(x_2, \mathcal{N}(A*0, A*Sigma*A.T + Sigma))

## Question 1.2: Determine the marginal distribution of x3, i.e. p(x3)

In [4]:
A = sp.MatrixSymbol("A", n, n)
Sigma = sp.MatrixSymbol("Sigma", n, n)

x1 = sp.MatrixSymbol("x_1", n, 1)
x2 = sp.MatrixSymbol("x_2", n, 1)
x3 = sp.MatrixSymbol("x_3", n, 1)

mapping = {
    y: x3,
    W: A,
    z: x2,
    Sigma_y: Sigma,
    mu_z: sp.ZeroMatrix(n, 1),
    Sigma_z: Sigma + A * Sigma*A.T
}
printit(mapping)

# Model

### Prior

Eq(x_2, \mathcal{N}(0, A*Sigma*A.T + Sigma))

### Likelihood

Eq(x_3, \mathcal{N}(A*x_2, Sigma))

...

Eq(\Sigma_(x_3|x_2), ((A*Sigma*A.T + Sigma)**(-1) + A.T*Sigma**(-1)*A)**(-1))

# Posterior mean eq. (3.37) Murphy1

Eq(\mu_(x_3|x_2), ((A*Sigma*A.T + Sigma)**(-1) + A.T*Sigma**(-1)*A)**(-1)*((A*Sigma*A.T + Sigma)**(-1)*0 + A.T*Sigma**(-1)*x_3))

# Marginal eq. (3.38) Murphy1

Eq(x_3, \mathcal{N}(A*0, A*(A*Sigma*A.T + Sigma)*A.T + Sigma))

## Question 1.3: Determine the conditional distribution of x3 given x1

In [5]:
A = sp.MatrixSymbol("A", n, n)
Sigma = sp.MatrixSymbol("Sigma", n, n)

x1 = sp.MatrixSymbol("x_1", n, 1)
x2 = sp.MatrixSymbol("x_2", n, 1)
x3 = sp.MatrixSymbol("x_3", n, 1)

mapping = {
    y: x3,
    W: A,
    z: x2,
    Sigma_y: Sigma,
    mu_z: A * x1,
    Sigma_z: Sigma
}
printit(mapping)

# Model

### Prior

Eq(x_2, \mathcal{N}(A*x_1, Sigma))

### Likelihood

Eq(x_3, \mathcal{N}(A*x_2, Sigma))

...

Eq(\Sigma_(x_3|x_2), (Sigma**(-1) + A.T*Sigma**(-1)*A)**(-1))

# Posterior mean eq. (3.37) Murphy1

Eq(\mu_(x_3|x_2), (Sigma**(-1) + A.T*Sigma**(-1)*A)**(-1)*(Sigma**(-1)*(A*x_1) + A.T*Sigma**(-1)*x_3))

# Marginal eq. (3.38) Murphy1

Eq(x_3, \mathcal{N}(A*(A*x_1), A*Sigma*A.T + Sigma))